# 🏭 07 — Filtro Industria 4.0 / Transizione 4.0

Carica tutti i file `classified_multiclass_aiuti_{year}.csv` (2014–2025) e produce un CSV unificato contenente i soli record i cui campi **Misura** (`TITOLO_MISURA`), **Titolo** (`TITOLO_PROGETTO`) o **Descrizione** (`DESCRIZIONE_PROGETTO`) menzionano:

- `Industria 4.0`
- `Transizione 4.0`
- `Impresa 4.0` (rebrand 2017)
- forme "Piano (Nazionale) Industria/Transizione/Impresa 4.0"

Il match generico `4.0` isolato è **escluso** per evitare falsi positivi (versioni software, release, ecc.).

**Output:** `data/export/industria_transizione_4_0.csv`

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

DATA_DIR = Path('../data')
OUT_PATH = DATA_DIR / 'export' / 'industria_transizione_4_0.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2014, 2026))

# Rendi importabile src/filter_4_0.py (necessario per ProcessPoolExecutor su Windows)
SRC_DIR = Path('..').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.filter_4_0 import PATTERN, PATTERN_STR, TEXT_COLS, filter_year  # noqa: E402

print('Setup completato ✓')

Setup completato ✓


## 2. Regex

Pattern unico, case-insensitive, ancorato a word boundary. Tollera separatori `-`, `_`, `.` e spazi multipli tra la parola chiave e `4.0`, e tra `4` e `0`.

Esempi di match: `Industria 4.0`, `industria4.0`, `Industria-4.0`, `transizione 4 . 0`, `Piano Nazionale Industria 4.0`, `Piano Impresa 4.0`.

Esempi di non-match: `versione 4.0`, `release 4.0`, `ISO 4.0`.

In [2]:
# Sanity check rapido
_tests_positive = [
    'Piano Industria 4.0', 'industria 4.0', 'Industria4.0', 'INDUSTRIA-4.0',
    'Transizione 4.0', 'transizione4.0', 'Piano Nazionale Transizione 4.0',
    'Impresa 4.0', 'Piano Impresa 4.0', 'Credito imposta Transizione 4.0 beni strumentali',
]
_tests_negative = [
    'versione 4.0', 'release 4.0', 'software 4.0', 'Windows 4.0', 'ISO 9001:4.0',
    'industria alimentare', 'transizione ecologica',
]

pos_ok = all(PATTERN.search(s) for s in _tests_positive)
neg_ok = all(not PATTERN.search(s) for s in _tests_negative)
print(f'Positive match : {pos_ok}')
print(f'Negative reject: {neg_ok}')
assert pos_ok and neg_ok, 'Regex self-test fallito'

Positive match : True
Negative reject: True


## 3. Caricamento e filtro per anno

Caricamento chunked per contenere l'uso di memoria (~24M righe totali). Per ogni chunk si applica il filtro regex sui tre campi testuali e si conservano solo le righe che combaciano.

In [3]:
import os
import time
from concurrent.futures import ProcessPoolExecutor

# Un processo per anno, limitato per non saturare l'I/O del disco
MAX_WORKERS = min(4, os.cpu_count() or 1)

t0 = time.perf_counter()
args_list = [(y, str(DATA_DIR.resolve())) for y in YEARS]

frames = []
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for year, df_y in zip(YEARS, ex.map(filter_year, args_list)):
        print(f'  {year}: {len(df_y):>8,} match')
        if not df_y.empty:
            frames.append(df_y)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
del frames
elapsed = time.perf_counter() - t0
print(f'\n✅ Totale record filtrati: {len(df):,}  ({elapsed:,.1f}s, {MAX_WORKERS} worker)')

  2014:        0 match
  2015:        0 match
  2016:        0 match
  2017:      177 match
  2018:    2,096 match
  2019:    1,958 match
  2020:    1,694 match
  2021:    2,343 match
  2022:    3,442 match
  2023:    3,302 match
  2024:    4,152 match
  2025:    1,459 match

✅ Totale record filtrati: 20,623  (79.9s, 4 worker)


## 4. Diagnostica

Verifica dove avviene il match (quale dei tre campi) e le misure più frequenti per sanity check.

In [4]:
if not df.empty:
    # Distribuzione per anno di concessione
    if 'ANNO' in df.columns:
        per_year = df.groupby('ANNO').size().rename('num_record')
        print('Record per anno:')
        print(per_year.to_string())
        print()

    # Match per campo
    print('Match per campo (un record può contribuire a più campi):')
    for col in TEXT_COLS:
        n = df[col].fillna('').astype(str).str.contains(PATTERN, regex=True, na=False).sum()
        print(f'  {col:<25} {n:>8,}')
    print()

    # Top 20 misure
    print('Top 20 TITOLO_MISURA:')
    print(df['TITOLO_MISURA'].value_counts().head(20).to_string())

Record per anno:
ANNO
2017     177
2018    2096
2019    1958
2020    1694
2021    2343
2022    3442
2023    3302
2024    4152
2025    1459

Match per campo (un record può contribuire a più campi):
  TITOLO_MISURA                5,628
  TITOLO_PROGETTO              7,725
  DESCRIZIONE_PROGETTO        15,278

Top 20 TITOLO_MISURA:
TITOLO_MISURA
Fondo regionale per la Crescita POR FESR CAMPANIA 2014/2020                                                                                                                                                                                                      4558
Regolamento per i fondi interprofessionali per la formazione continua per la concessioni di aiuti di stato esentati ai sensi del regolamento CE n.651/2014 e in regime de minimis ai sensi del regolamento CE n.1407/2013                                        1842
Bando Voucher Digitali Impresa 4.0 - anno 2021                                                                                     

## 5. Export

In [5]:
if df.empty:
    print('⚠ Nessun record da esportare.')
else:
    df.to_csv(OUT_PATH, index=False)
    size_mb = OUT_PATH.stat().st_size / 1e6
    print(f'✅ Scritto: {OUT_PATH}')
    print(f'   Righe  : {len(df):,}')
    print(f'   Colonne: {len(df.columns)}')
    print(f'   Size   : {size_mb:,.2f} MB')

✅ Scritto: ..\data\export\industria_transizione_4_0.csv
   Righe  : 20,623
   Colonne: 25
   Size   : 11.96 MB
